# TruthLens UA — ISOT Fake News Detection + MLflow
Training LinearSVC on ISOT dataset (EN), save pipeline to `artifacts/best_model.joblib`.
Student: 102012dl | Neoversity MSCS DS&DA 2026

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report

os.makedirs("artifacts", exist_ok=True)
os.makedirs("data", exist_ok=True)
print("Imports OK")

In [ ]:
# Load ISOT: expect data/Fake.csv and data/True.csv (or data/isot_*.csv)
# Download from: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
# Or run: python scripts/download_datasets.py

def load_isot(data_dir="data"):
    fake_path = os.path.join(data_dir, "Fake.csv")
    true_path = os.path.join(data_dir, "True.csv")
    if os.path.exists(fake_path) and os.path.exists(true_path):
        fake = pd.read_csv(fake_path)
        true = pd.read_csv(true_path)
        fake["label"] = "FAKE"
        true["label"] = "REAL"
        text_col = "text" if "text" in fake.columns else ("title" if "title" in fake.columns else fake.columns[1])
        df = pd.concat([fake[[text_col, "label"]], true[[text_col, "label"]]], ignore_index=True)
        df = df.dropna(subset=[text_col])
        df[text_col] = df[text_col].astype(str)
        return df[text_col], df["label"], text_col
    # Fallback: minimal sample for pipeline test
    texts = ["Breaking: fake news story.", "Reuters reports official data."] * 50
    labels = ["FAKE", "REAL"] * 50
    return pd.Series(texts), pd.Series(labels), "text"

X_series, y_series, _ = load_isot()
X, y = X_series.tolist(), y_series.tolist()
print(f"Samples: {len(X)}, labels: {pd.Series(y).value_counts().to_dict()}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", len(X_train), "Test:", len(X_test))

In [ ]:
mlflow.set_experiment("truthlens-isot-linear-svc")

with mlflow.start_run(run_name="LinearSVC_ISOT_50k"):
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2)),
        ("clf", LinearSVC(max_iter=10000, C=1.0)),
    ])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    f1 = f1_score(y_test, preds, average="weighted")
    acc = accuracy_score(y_test, preds)
    mlflow.log_metrics({"f1": f1, "accuracy": acc})
    mlflow.sklearn.log_model(pipeline, "model")
    print("Model classes:", pipeline.classes_)
    print(classification_report(y_test, preds))
    print(f"F1={f1:.4f} Accuracy={acc:.4f}")

In [ ]:
joblib.dump(pipeline, "artifacts/best_model.joblib")
print("Model saved to artifacts/best_model.joblib")
print("Classes:", pipeline.classes_)

In [ ]:
# Visualization: confusion matrix and summary table
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

labels_order = sorted(set(y_test))
cm = confusion_matrix(y_test, preds, labels=labels_order)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels_order, yticklabels=labels_order, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("ISOT — Confusion Matrix (LinearSVC)")
plt.tight_layout()
plt.show()

metrics_df = pd.DataFrame([
    {"metric": "F1_weighted", "value": f1},
    {"metric": "Accuracy", "value": acc},
])
metrics_df["value"] = metrics_df["value"].round(4)
metrics_df